# 셀프 쿼리(Self-querying)

`SelfQueryRetriever` 는 자체적으로 질문을 생성하고 해결할 수 있는 기능을 갖춘 검색 도구입니다. 

이는 사용자가 제공한 자연어 질의를 바탕으로, `query-constructing` LLM chain을 사용해 구조화된 질의를 만듭니다. 그 후, 이 구조화된 질의를 기본 벡터 데이터 저장소(VectorStore)에 적용하여 검색을 수행합니다.

이 과정을 통해, `SelfQueryRetriever` 는 단순히 사용자의 입력 질의를 저장된 문서의 내용과 의미적으로 비교하는 것을 넘어서, 사용자의 질의에서 문서의 메타데이터에 대한 **필터를 추출** 하고, 이 필터를 실행하여 관련된 문서를 찾을 수 있습니다. 

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## 샘플 데이터 생성
화장품 상품의 설명과 메타데이터를 기반으로 유사도 검색이 가능한 벡터 저장소를 구축합니다.

In [2]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 화장품의 설명과 메타데이터 생성
docs = [
    Document(
            page_content="수분 가득한 히알루론산 세럼으로 피부 속 깊은 곳까지 수분을 공급합니다.",
            metadata={"year": 2024, "category": "스킨케어", "user_rating": 4.7},
        ),
        Document(
            page_content="24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.",
            metadata={"year": 2023, "category": "메이크업", "user_rating": 4.5},
        ),
        Document(
            page_content="식물성 성분으로 만든 저자극 클렌징 오일, 메이크업과 노폐물을 부드럽게 제거합니다.",
            metadata={"year": 2023, "category": "클렌징", "user_rating": 4.8},
        ),
        Document(
            page_content="비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.",
            metadata={"year": 2023, "category": "스킨케어", "user_rating": 4.6},
        ),
        Document(
            page_content="롱래스팅 립스틱, 선명한 발색과 촉촉한 사용감으로 하루종일 편안하게 사용 가능합니다.",
            metadata={"year": 2024, "category": "메이크업", "user_rating": 4.4},
        ),
        Document(
            page_content="자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.",
            metadata={"year": 2024, "category": "선케어", "user_rating": 4.9},
        ),
]

# Chroma 벡터 저장소 생성
vectorstore = Chroma.from_documents(
    docs, OpenAIEmbeddings(model="text-embedding-3-small")
)


## SelfQueryRetriever

이제 retriever를 인스턴스화할 수 있습니다. 이를 위해서는 문서가 지원하는 **메타데이터 필드** 와 문서 내용에 대한 **간단한 설명을 미리 제공** 해야 합니다.

`AttributeInfo` 클래스를 사용하여 화장품 메타데이터 필드에 대한 정보를 정의합니다.

- 카테고리(`category`): 문자열 타입, 화장품의 카테고리를 나타내며 ['스킨케어', '메이크업', '클렌징', '선케어'] 중 하나의 값을 가집니다.
- 연도(`year`): 정수 타입, 화장품이 출시된 연도를 나타냅니다.
- 사용자 평점(`user_rating`): 실수 타입, 1-5 범위의 사용자 평점을 나타냅니다.


In [3]:
from langchain_classic.chains.query_constructor.base import AttributeInfo

# 메타데이터 필드 정보 
metadata_field_info = [
    AttributeInfo(
        name="category",
        description="The category of the cosmetic product. One of ['스킨케어', '메이크업', '클렌징', '선케어']",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the cosmetic product was released",
        type="integer",
    ),
    AttributeInfo(
        name="user_rating",
        description="A user rating for cosmetic product, ranging from 1 to 5",
        type="float",
    ),
]

`SelfQueryRetriever.from_llm()` 메서드를 사용하여 `retriever` 객체를 생성합니다.

- `llm`: 언어 모델
- `vectorstore`: 벡터 저장소
- `document_contents`: 문서들의 내용 설명
- `metadata_field_info`: 메타데이터 필드 정보


In [6]:
import lark

print(lark.__version__)

1.3.1


In [30]:
import inspect
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.query_constructors.chroma import ChromaTranslator
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5", temperature=0)

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="Brief summary of a cosmetic product",
    metadata_field_info=metadata_field_info,
    structured_query_translator=ChromaTranslator(),
)

## Query 테스트

In [17]:
print(docs[0].metadata)
print(type(docs[0].metadata["user_rating"]))

{'year': 2024, 'category': '스킨케어', 'user_rating': 4.7}
<class 'float'>


In [18]:
print(vectorstore.get()["metadatas"])

[{'category': '스킨케어', 'year': 2024, 'user_rating': 4.7}, {'user_rating': 4.5, 'category': '메이크업', 'year': 2023}, {'year': 2023, 'category': '클렌징', 'user_rating': 4.8}, {'year': 2023, 'user_rating': 4.6, 'category': '스킨케어'}, {'category': '메이크업', 'user_rating': 4.4, 'year': 2024}, {'user_rating': 4.9, 'year': 2024, 'category': '선케어'}]


In [19]:
query_constructor = retriever.query_constructor

result = query_constructor.invoke(
    "평점이 4.8 이상인 제품을 추천해주세요"
)

print(result)

query=' ' filter=Comparison(comparator=<Comparator.GTE: 'gte'>, attribute='user_rating', value=4.8) limit=None


In [20]:
translated = ChromaTranslator().visit_comparison(
    result.filter
)

print(translated)

{'user_rating': {'$gte': 4.8}}


In [21]:
print(type(result.filter.value))
print(result.filter.value)

<class 'float'>
4.8


In [22]:
print(type(result.filter.value))
print(repr(result.filter.value))

<class 'float'>
4.8


In [23]:
retriever.invoke("평점이 4.8 이상인 제품을 추천해주세요")

[Document(id='728dea13-b361-4c37-a7e6-18d79af79854', metadata={'user_rating': 4.9, 'year': 2024, 'category': '선케어'}, page_content='자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.'),
 Document(id='ff46beb7-3847-4cc4-b90c-193e52f50cc2', metadata={'year': 2023, 'user_rating': 4.8, 'category': '클렌징'}, page_content='식물성 성분으로 만든 저자극 클렌징 오일, 메이크업과 노폐물을 부드럽게 제거합니다.')]

In [24]:
# Self-query 검색
retriever.invoke("2023년에 출시된 상품을 추천해주세요")

[Document(id='51e9e180-fa89-4f55-84cb-f441fa1c3bec', metadata={'user_rating': 4.5, 'year': 2023, 'category': '메이크업'}, page_content='24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.'),
 Document(id='af288c63-cd98-401d-b09e-bd67a428d615', metadata={'user_rating': 4.6, 'year': 2023, 'category': '스킨케어'}, page_content='비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.'),
 Document(id='ff46beb7-3847-4cc4-b90c-193e52f50cc2', metadata={'category': '클렌징', 'year': 2023, 'user_rating': 4.8}, page_content='식물성 성분으로 만든 저자극 클렌징 오일, 메이크업과 노폐물을 부드럽게 제거합니다.')]

In [25]:
# Self-query 검색
retriever.invoke("카테고리가 선케어인 상품을 추천해주세요")

[Document(id='728dea13-b361-4c37-a7e6-18d79af79854', metadata={'category': '선케어', 'user_rating': 4.9, 'year': 2024}, page_content='자외선 차단 기능이 있는 톤업 선크림, SPF50+/PA++++ 높은 자외선 차단 지수로 피부를 보호합니다.')]

In [26]:
# Self-query 검색
retriever.invoke(
    "카테고리가 메이크업인 상품 중에서 평점이 4.5 이상인 상품을 추천해주세요"
)

[Document(id='51e9e180-fa89-4f55-84cb-f441fa1c3bec', metadata={'user_rating': 4.5, 'year': 2023, 'category': '메이크업'}, page_content='24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.')]

`k`는 가져올 문서의 수를 의미합니다.

`SelfQueryRetriever`를 사용하여 `k`를 지정할 수도 있습니다. 이는 생성자에 `enable_limit=True`를 전달하여 수행할 수 있습니다.


In [35]:
import inspect
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.query_constructors.chroma import ChromaTranslator
from langchain_openai import ChatOpenAI

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="Brief summary of a cosmetic product",
    metadata_field_info=metadata_field_info,
    enable_limit=True,   # 검색 결과 제한 기능 활성화
    search_kwargs={"k":2},
    structured_query_translator=ChromaTranslator(),
)

In [36]:
# Self-query 검색
retriever.invoke("2023년에 출시된 상품을 추천해주세요")

[Document(id='51e9e180-fa89-4f55-84cb-f441fa1c3bec', metadata={'year': 2023, 'user_rating': 4.5, 'category': '메이크업'}, page_content='24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.'),
 Document(id='af288c63-cd98-401d-b09e-bd67a428d615', metadata={'category': '스킨케어', 'user_rating': 4.6, 'year': 2023}, page_content='비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.')]

하지만 코드로 명시적으로 `search_kwargs`를 지정하지 않고 query 에서 `1개, 2개` 등의 숫자를 사용하여 검색 결과를 제한할 수 있습니다.


In [37]:
import inspect
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.query_constructors.chroma import ChromaTranslator
from langchain_openai import ChatOpenAI

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    document_contents="Brief summary of a cosmetic product",
    metadata_field_info=metadata_field_info,
    enable_limit=True,  # 검색 결과 제한 기능 활성화
    structured_query_translator=ChromaTranslator(),
)

In [38]:
# Self-query 검색
retriever.invoke("2023년에 출시된 상품 1개를 추천해주세요")

[Document(id='51e9e180-fa89-4f55-84cb-f441fa1c3bec', metadata={'category': '메이크업', 'user_rating': 4.5, 'year': 2023}, page_content='24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.')]

In [44]:
result = retriever.query_constructor.invoke("2023년에 출시된 상품 1개를 추천해주세요")
print(result)   # query, filter, limit 확인


query=' ' filter=Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='year', value=2023) limit=1


In [39]:
# Self-query 검색
retriever.invoke("2023년에 출시된 상품 2개를 추천해주세요")

[Document(id='51e9e180-fa89-4f55-84cb-f441fa1c3bec', metadata={'user_rating': 4.5, 'category': '메이크업', 'year': 2023}, page_content='24시간 지속되는 매트한 피니시의 파운데이션, 모공을 커버하고 자연스러운 피부 표현이 가능합니다.'),
 Document(id='af288c63-cd98-401d-b09e-bd67a428d615', metadata={'year': 2023, 'category': '스킨케어', 'user_rating': 4.6}, page_content='비타민 C 함유 브라이트닝 크림, 칙칙한 피부톤을 환하게 밝혀줍니다.')]

In [40]:
result = retriever.query_constructor.invoke("2023년에 출시된 상품 2개를 추천해주세요")
print(result)
# query=' ' filter=Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='year', value=2023) limit=2

query=' ' filter=Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='year', value=2023) limit=2


In [41]:
translated = ChromaTranslator().visit_structured_query(result)
print(translated)
# (' ', {'filter': {'year': {'$eq': 2023}}, 'k': 2, ...} 같은 형태)


(' ', {'filter': {'year': {'$eq': 2023}}})


In [43]:
print(retriever.query_constructor.get_prompts()[0].format(query="2023년에 출시된 상품 2개를 추천해주세요"))

Your goal is to structure the user's query to match the request schema provided below.

<< Structured Request Schema >>
When responding use a markdown code snippet with a JSON object formatted in the following schema:

```json
{
    "query": string \ text string to compare to document contents
    "filter": string \ logical condition statement for filtering documents
    "limit": int \ the number of documents to retrieve
}
```

The query string should contain only text that is expected to match the contents of documents. Any conditions in the filter should not be mentioned in the query as well.

A logical condition statement is composed of one or more comparison and logical operation statements.

A comparison statement takes the form: `comp(attr, val)`:
- `comp` (eq | ne | gt | gte | lt | lte): comparator
- `attr` (string):  name of attribute to apply the comparison to
- `val` (string): is the comparison value

A logical operation statement takes the form `op(statement1, statement2, ..

## 더 깊게 들어가기

내부에서 어떤 일이 일어나는지 확인하고 더 많은 사용자 정의 제어를 하기 위해, 우리는 retriever를 처음부터 재구성할 수 있습니다.

이 과정은 `query-construction chain` 을 생성하는 것부터 시작합니다.

### `query_constructor` chain 생성

구조화된 쿼리를 생성하는 `query_constructor` chain 을 생성합니다.
`get_query_constructor_prompt` 함수를 사용하여 쿼리 생성기 프롬프트를 가져옵니다.

`prompt.format()` 메서드를 사용하여 `query` 매개변수에 "dummy question" 문자열을 전달하고, 그 결과를 출력하여 Prompt 내용을 확인해 보겠습니다.


`query_constructor.invoke()` 메서드를 호출하여 주어진 쿼리에 대한 처리를 수행합니다.


Self-query retriever의 핵심 요소는 query constructor입니다. 훌륭한 검색 시스템을 만들기 위해서는 query constructor가 잘 작동하도록 해야 합니다.

이를 위해서는 **프롬프트(Prompt), 프롬프트 내의 예시, 속성 설명 등을 조정** 해야 합니다.

### 구조화된 쿼리 변환기(Structured Query Translator)를 사용하여 구조화된 쿼리로 변환

다음으로 중요한 요소는 structured query translator입니다. 

이는 일반적인 `StructuredQuery` 객체를 사용 중인 vector store의 구문에 맞는 메타데이터 필터로 변환하는 역할을 담당합니다.